In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import os
import time

# 🔥 TELEGRAM FUNCTION
def send_telegram_message(message):
    TOKEN = "YOUR_BOT_TOKEN"
    CHAT_ID = "YOUR_CHAT_ID"

    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    data = {
        "chat_id": CHAT_ID,
        "text": message
    }

    try:
        requests.post(url, data=data)
    except:
        print("⚠️ Telegram failed")

# 🔹 PRODUCTS
products = [
    {"name": "Mobile Phone", "url": "https://www.amazon.in/dp/B0GP8XYL7H"},
    {"name": "Headphones", "url": "https://www.amazon.in/dp/B0F4N3T2PH"},
    {"name": "Shoes", "url": "https://www.amazon.in/dp/B07QC2CPWP"},
    {"name": "T-Shirt", "url": "https://www.amazon.in/dp/B0F5GX46NX"},
    {"name": "Laptop", "url": "https://www.amazon.in/dp/B0FRM3G9JF"},
    {"name": "Backpack", "url": "https://www.amazon.in/dp/B097JJ2CK6"}
]

# 🔹 HEADERS
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-IN,en;q=0.9"
}

# 🔹 LOAD OLD DATA
if os.path.exists("data_price.csv"):
    old_df = pd.read_csv("data_price.csv")
else:
    old_df = pd.DataFrame()

all_data = []

# 🔁 LOOP
for product in products:
    print(f"Scraping {product['name']}...")

    try:
        response = requests.get(product["url"], headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"❌ Failed: {product['name']}")
            continue

        soup = BeautifulSoup(response.content, "html.parser")

        # 🔹 TITLE
        title = soup.select_one("#productTitle")
        title_text = title.get_text().strip() if title else "Not Found"

        # 🔹 PRICE (SAFE EXTRACTION)
        price = soup.select_one(".a-price .a-offscreen")

        if price:
            price_text = price.get_text().strip()

            if price_text:
                try:
                    price_value = float(price_text.replace("₹", "").replace(",", ""))
                except:
                    print(f"⚠️ Conversion error: {product['name']} → '{price_text}'")
                    continue
            else:
                print(f"⚠️ Empty price: {product['name']}")
                continue
        else:
            print(f"⚠️ Price not found: {product['name']}")
            continue

        # ⚠️ BLOCK CHECK
        if title_text == "Not Found":
            print(f"⚠️ Blocked or not loaded: {product['name']}")

        # 🔥 PRICE ALERT
        if not old_df.empty:
            prev = old_df[old_df["product_name"] == product["name"]]

            if not prev.empty:
                last_price = prev.iloc[-1]["price"]

                if price_value > 0 and price_value < last_price and (last_price - price_value) > 100:
                    message = f"""🔥 PRICE DROP!

Product: {product['name']}
Old Price: ₹{last_price}
New Price: ₹{price_value}
"""

                    print(message)
                    send_telegram_message(message)

        # 📊 STORE DATA
        data = {
            "product_name": product["name"],
            "title": title_text,
            "price": price_value,
            "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        all_data.append(data)

    except Exception as e:
        print(f"❌ Error: {product['name']} → {e}")

    time.sleep(2)

# 💾 SAVE
df = pd.DataFrame(all_data)

file_exists = os.path.isfile("data_price.csv")

df.to_csv("data_price.csv", mode="a", header=not file_exists, index=False)

# 📝 LOG (for Task Scheduler debugging)
with open("log.txt", "a") as f:
    f.write(f"Ran at {datetime.now()}\n")

print("✅ All data saved successfully!")
print(df)

Scraping Mobile Phone...
Scraping Headphones...
🔥 PRICE DROP!

Product: Headphones
Old Price: ₹2699.0
New Price: ₹2499.0

Scraping Shoes...
⚠️ Empty price: Shoes
Scraping T-Shirt...
⚠️ Empty price: T-Shirt
Scraping Laptop...
🔥 PRICE DROP!

Product: Laptop
Old Price: ₹71979.0
New Price: ₹65900.0

Scraping Backpack...
✅ All data saved successfully!
   product_name                                              title    price  \
0  Mobile Phone  iQOO Z11x 5G (Prismatic Green, 8GB RAM, 128 GB...  20999.0   
1    Headphones  ZEBRONICS SILENCIO 111, Wireless Headphone, Hy...   2499.0   
2        Laptop  HP Omnibook 5 OLED (Previously Pavilion), Snap...  65900.0   
3      Backpack  Safari Omega spacious/large laptop backpack wi...    619.0   

                  date  
0  2026-04-05 00:37:59  
1  2026-04-05 00:38:04  
2  2026-04-05 00:38:15  
3  2026-04-05 00:38:20  
